In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_33_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 34 ###

def grab_subset_of_data_46(original_df, question_of_interest):
    # select and rename only the columns containing the question, drop rows that are all NA
    cols = [c for c in original_df.columns if question_of_interest in c]
    rename_map = {c: c.rsplit('-', 1)[-1].lstrip() for c in cols}
    subset = original_df.loc[:, cols].rename(columns=rename_map)
    return subset.dropna(how='all', subset=rename_map.values())


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
    question_of_interest,
    include_2017=False
):
    # pull the same question from each year's DataFrame and tag with year, then concat
    mapping = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_df_2019_cell10,
        '2018': responses_df_2018_cell10,
        '2017': responses_df_2017
    }
    years = ['2018','2019','2020','2021','2022']
    if include_2017:
        years.insert(0, '2017')
    df_list = []
    for y in years:
        df = grab_subset_of_data_46(mapping[y], question_of_interest)
        df['year'] = y
        df_list.append(df)
    df_combined = pd.concat(df_list, ignore_index=True)
    df_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_counts


def convert_df_of_counts_to_percentages_46(df, df_counts):
    # divide each year's counts by the total responses in that year, then *100
    dfc = df_counts.set_index('year')
    totals = df['year'].value_counts().sort_index()
    dfc = dfc.div(totals, axis=0).mul(100)
    return dfc.reset_index()


# first, rename 2018's question columns to match the others
old = 'What machine learning frameworks have you used in the past 5 years?'
new = 'Which of the following machine learning frameworks do you use on a regular basis?'
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(old, new, regex=False)

question = new
# combine all years 2018–2022 (no 2017)
ml_df_combined, ml_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(question)
)

# consolidate multiple‐response variants into single flag columns
col_groups = {
    'TensorFlow ': ['TensorFlow', 'TensorFlow '],
    'Keras ':      ['Keras',      'Keras '],
    'PyTorch ':    ['PyTorch',    'PyTorch '],
    'Scikit-learn ': [c for c in ml_df_combined.columns if c.lower().startswith('scikit')]
}
for new_col, olds in col_groups.items():
    # only use the old columns that actually exist
    existing_old = [c for c in olds if c in ml_df_combined.columns]
    if existing_old:
        mask = ml_df_combined[existing_old].notna().any(axis=1)
        # assign the new_col value (with trailing space) or NaN
        ml_df_combined[new_col] = np.where(mask, new_col, np.nan)
# drop only the original old columns, not the newly created ones
to_drop = [c for olds in col_groups.values() for c in olds if c in ml_df_combined.columns]
ml_df_combined_cell46 = ml_df_combined.drop(columns=to_drop)

# recompute counts and convert to percentages
ml_df_combined_counts_2 = ml_df_combined_cell46.groupby('year').count().reset_index()
ml_df_combined_percentages = convert_df_of_counts_to_percentages_46(
    ml_df_combined_cell46,
    ml_df_combined_counts_2
)

# ensure all expected columns are present (fill missing with 0)
expected = ['Scikit-learn ', 'TensorFlow ', 'Keras ', 'PyTorch ', 'None', 'Other']
ml_df_combined_percentages = ml_df_combined_percentages.reindex(
    columns=['year'] + expected,
    fill_value=0
)

# reshape for plotting/table
df_cell46 = ml_df_combined_percentages.melt(
    id_vars=['year'],
    value_vars=expected
)
# sort and rename
df_cell46 = df_cell46.sort_values(by=['year', 'value'], ascending=True)
df_cell46.rename(columns={'variable': ''}, inplace=True)

df_cell46.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30 entries, 5 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    30 non-null     object 
 1           30 non-null     object 
 2   value   30 non-null     float64
dtypes: float64(1), object(2)
memory usage: 960.0+ bytes
CPU times: user 70.4 ms, sys: 4 ms, total: 74.4 ms
Wall time: 74.5 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_34_try_1.pickle